# Standalone DQN Notebook

Self-contained PyTorch demonstration of the served DQN contract: state vector, action Q-values, reward shaping, replay buffer, target network, soft update, and checkpoint-compatible `state_dict` training.


In [ ]:
from collections import deque
import hashlib
import math
import random

import numpy as np
import torch
from torch import nn

EMBED_DIM = 16
FEATURE_DIM = EMBED_DIM + 6
ACTIONS = ['python', 'sql', 'machine learning', 'dashboard', 'public speaking', 'ux research']
torch.manual_seed(7)
random.seed(7)


def projected_embedding(seed, dim=EMBED_DIM):
    raw = np.frombuffer(hashlib.blake2b(seed.encode('utf-8'), digest_size=32).digest(), dtype=np.uint8).astype(np.float32)
    values = np.tile(raw, math.ceil(dim / raw.size))[:dim]
    values = values / 127.5 - 1.0
    norm = np.linalg.norm(values)
    return values / norm if norm else values


def encode_state(user_id, job, context=None):
    context = context or {}
    embedding = projected_embedding(f"job:{job['id']}:{job.get('title', '')}")
    dense = np.array([
        float(job.get('sbert_score', 0.0)),
        float(job.get('ncf_score', 0.0)),
        math.log1p(len(context.get('history', []))) / 5.0,
        math.log1p(context.get('interaction_count', 0)) / 5.0,
        min(1.0, len(job.get('description', '')) / 1500.0),
        1.0,
    ], dtype=np.float32)
    return np.concatenate([embedding, dense]).astype(np.float32)

state = encode_state('u-1', {'id': 'job-data', 'title': 'Data Scientist', 'sbert_score': 0.8, 'ncf_score': 0.7})
print(state.shape)


In [ ]:
class TinyQNetwork(nn.Module):
    def __init__(self, state_dim=FEATURE_DIM, n_actions=len(ACTIONS), hidden=64):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, states):
        return self.layers(states)


policy_net = TinyQNetwork()
target_net = TinyQNetwork()
target_net.load_state_dict(policy_net.state_dict())


def q_values(network, state_vector):
    with torch.inference_mode():
        tensor = torch.tensor(np.asarray([state_vector]), dtype=torch.float32)
        return network(tensor)[0].detach().numpy()


def top_n_actions(state_vector, n=3):
    values = q_values(policy_net, state_vector)
    order = np.argsort(values)[::-1][:n]
    return [(ACTIONS[index], float(values[index])) for index in order]


def rerank_top_n(user_id, jobs, n=3):
    rows = []
    for job in jobs:
        values = q_values(policy_net, encode_state(user_id, job))
        rows.append((job['id'], float(np.max(values))))
    return sorted(rows, key=lambda row: row[1], reverse=True)[:n]

print(top_n_actions(state, 3))
print(rerank_top_n('u-1', [{'id': 'a'}, {'id': 'b'}, {'id': 'c'}], 3))


In [ ]:
def reward(event, same_domain=True):
    base = {'skip': -0.5, 'view': 0.5, 'click': 1.0, 'save': 0.85, 'apply': 1.0}[event]
    if not same_domain:
        base *= 0.2
    return max(-1.0, min(1.0, base))

assert reward('apply') == 1.0
assert reward('apply', same_domain=False) == 0.2
assert reward('skip') < 0
print({event: reward(event) for event in ['skip', 'view', 'click', 'save', 'apply']})


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=1000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward_value, next_state, done):
        self.buffer.append((state, action, reward_value, next_state, done))

    def sample(self, batch_size):
        return random.sample(list(self.buffer), min(batch_size, len(self.buffer)))


def soft_update(target_network, online_network, tau=0.05):
    with torch.no_grad():
        for target_param, online_param in zip(target_network.parameters(), online_network.parameters()):
            target_param.data.mul_(1.0 - tau).add_(online_param.data, alpha=tau)


buffer = ReplayBuffer()
optimizer = torch.optim.AdamW(policy_net.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
next_state = encode_state('u-1', {'id': 'job-dashboard', 'title': 'Dashboard Analyst', 'sbert_score': 0.7, 'ncf_score': 0.6})
buffer.push(state, ACTIONS.index('machine learning'), reward('click'), next_state, False)

batch = buffer.sample(8)
states = torch.tensor(np.asarray([row[0] for row in batch]), dtype=torch.float32)
actions = torch.tensor([row[1] for row in batch], dtype=torch.long)
rewards = torch.tensor([row[2] for row in batch], dtype=torch.float32)
next_states = torch.tensor(np.asarray([row[3] for row in batch]), dtype=torch.float32)
dones = torch.tensor([row[4] for row in batch], dtype=torch.bool)

pred = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
with torch.no_grad():
    target = rewards + 0.92 * target_net(next_states).max(dim=1).values * (~dones).float()
loss = loss_fn(pred, target)
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
soft_update(target_net, policy_net)

assert len(buffer.sample(8)) == 1
assert loss.item() >= 0.0
print('replay size', len(buffer.buffer), 'loss', round(float(loss.item()), 4))
